In [1]:
%load_ext autoreload
%autoreload 2

import yfinance as yf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import sys

# 1. Get the absolute path of the current script's directory
current_dir = os.getcwd()

# 2. Get the path of the parent directory
parent_dir = os.path.dirname(current_dir)

# 3. Add the parent directory to the top of sys.path
sys.path.insert(0, parent_dir)

import baselines,metrics,features,models


In [2]:
print(metrics.__file__)
print([name for name in dir(metrics) if not name.startswith("_")])


c:\Users\user\Desktop\Projects\Time-series ML\metrics.py
['QLIKE', 'RMSE', 'np', 'pd', 'plt']


In [3]:
print(baselines.__file__)
print([name for name in dir(baselines) if not name.startswith("_")])


c:\Users\user\Desktop\Projects\Time-series ML\baselines.py
['arch_model', 'ewm', 'garch', 'naive', 'np', 'pd', 'rolling']


In [4]:
data = yf.download("^GSPC", start="1990-01-01", auto_adjust=True)


log_returns = np.log(data["Close"]["^GSPC"]).diff().iloc[1:]


sq_returns = np.square(log_returns)
vol_of_5days = np.sqrt(sq_returns.rolling(5).mean()).shift(-5).iloc[:log_returns.shape[0] - 5]


train = vol_of_5days.loc["1990" : "2018"]
val = vol_of_5days.loc["2019" : "2022"]
test = vol_of_5days.loc["2023" : ]


print(train.head())
print(train.tail())
print(val.head())
print(val.tail())
print(test.head())
print(test.tail())


[*********************100%***********************]  1 of 1 completed

Date
1990-01-03    0.008669
1990-01-04    0.007915
1990-01-05    0.012972
1990-01-08    0.013387
1990-01-09    0.013250
Name: ^GSPC, dtype: float64
Date
2018-12-24    0.022317
2018-12-26    0.012456
2018-12-27    0.019198
2018-12-28    0.019442
2018-12-31    0.019553
Name: ^GSPC, dtype: float64
Date
2019-01-02    0.019630
2019-01-03    0.016240
2019-01-04    0.005983
2019-01-07    0.005621
2019-01-08    0.005976
Name: ^GSPC, dtype: float64
Date
2022-12-23    0.009848
2022-12-27    0.010246
2022-12-28    0.010157
2022-12-29    0.012052
2022-12-30    0.012003
Name: ^GSPC, dtype: float64
Date
2023-01-03    0.012269
2023-01-04    0.013109
2023-01-05    0.012113
2023-01-06    0.006921
2023-01-09    0.006972
Name: ^GSPC, dtype: float64
Date
2026-10-13    0.007485
2026-10-14    0.006151
2026-10-15    0.008080
2026-10-16    0.006823
2026-10-19    0.007381
Name: ^GSPC, dtype: float64


In [5]:
print(data.columns)


MultiIndex([( 'Close', '^GSPC'),
            (  'High', '^GSPC'),
            (   'Low', '^GSPC'),
            (  'Open', '^GSPC'),
            ('Volume', '^GSPC')],
           names=['Price', 'Ticker'])


In [6]:
pred = pd.Series([1])
act = pd.Series([0.5])

print(metrics.QLIKE(pred,act))

0.6362943611198906


In [7]:
print(val.head())

Date
2019-01-02    0.019630
2019-01-03    0.016240
2019-01-04    0.005983
2019-01-07    0.005621
2019-01-08    0.005976
Name: ^GSPC, dtype: float64


In [8]:
import inspect
print(baselines.__file__)
print(inspect.getsource(baselines.garch))

c:\Users\user\Desktop\Projects\Time-series ML\baselines.py
def garch(returns: pd.Series, cutoff: str) -> pd.Series:
    """GARCH(1,1) forecast of next-week volatility (typical daily move).

    Like EWMA, but reaction (alpha), memory (beta) and a long-run anchor (omega)
    are learned from data before `cutoff`, instead of being fixed.
    """
    r100 = returns * 100  # arch fits more reliably on values around 1
    model = arch_model(r100, mean="Zero", vol="GARCH", p=1, q=1)
    res = model.fit(last_obs=cutoff, disp="off")              # learn omega, alpha, beta before cutoff
    fc = res.forecast(horizon=5, start=cutoff, reindex=False)  # 5 daily variances per date
    daily_var = fc.variance / 100**2                          # undo the scaling
    return np.sqrt(daily_var.mean(axis=1))  



In [9]:
print(type(data))
garch = baselines.garch(log_returns,"2019-01-02")

<class 'pandas.DataFrame'>


In [10]:
naive = baselines.naive(log_returns)
rolling = baselines.rolling(log_returns)
ewm = baselines.ewm(log_returns)

print(metrics.QLIKE(garch,val)) 
print(metrics.QLIKE(naive,val))
print(metrics.QLIKE(rolling,val))
print(metrics.QLIKE(ewm,val))

0.4935176358500769
1.0081561305501578
0.6309387687914793
0.5560741281683151


In [11]:
df = features.creater_features(data)

train = df.loc["1990" : "2018"]
val = df.loc["2019" : "2022"]
test = df.loc["2023" : ]

print(train.shape)

train = train[train["today_vol"] != 0]

print(train.shape)


(7285, 4)
(7280, 4)


In [12]:
q = models.HAR(train,val)  
qlog = models.HAR_LOG(train,val)
print(metrics.QLIKE(q,val["target"]))
print(metrics.QLIKE(qlog,val["target"]))
print(metrics.RMSE(q,val["target"]))
print(metrics.RMSE(garch,val["target"]))
print(metrics.RMSE(qlog,val["target"]))

0.5453101982403935
0.5613287814576948
0.006687369621191603
0.006900738155342089
0.006906259875082031


In [13]:
print((qlog < val["target"]).mean())   # share of days under-predicted
print(qlog.mean(), val["target"].mean())

0.4126984126984127
0.010815706859956963 0.011109548164003918


In [14]:
cols = ["today_vol", "past_5_days", "past_22_days"]

for leaf in [5,10,20,40,50,65,100,200]:
    rf = models.random_forest(train,val,cols,leaf)
    print(metrics.QLIKE(rf,val["target"]))



for leaf in [5,10,20,40,50,65,100,200]:
    rf = models.random_forest(train,val,cols,leaf)
    print(metrics.RMSE(rf,val["target"]))




0.6157433472612606
0.5997385641705301
0.5785394129582775
0.5599721896505986
0.5518876034862485
0.5473703989384533
0.5649866415527226
0.5859359691577624
0.006767328332560006
0.0067403393668088
0.006660437788604903
0.0064731006547313385
0.006448497508947653
0.006493395903417859
0.006959705396866156
0.007343778563362993


In [15]:

    for lr in [0.01,0.05,0.1]:
        for depth in [1,2,3,4,5]:
            xgb = models.XGBOOST(train,val,cols,depth,lr)
            score = metrics.QLIKE(xgb,val["target"])

            print(f"for lr={lr}, depth = {depth}, the score is {score}")

stopped at tree 580
for lr=0.01, depth = 1, the score is 0.517239835520668
stopped at tree 243
for lr=0.01, depth = 2, the score is 0.5178609380391591
stopped at tree 196
for lr=0.01, depth = 3, the score is 0.5299469184285647
stopped at tree 155
for lr=0.01, depth = 4, the score is 0.54192772077357
stopped at tree 141
for lr=0.01, depth = 5, the score is 0.5552423029229998
stopped at tree 105
for lr=0.05, depth = 1, the score is 0.5171054415898831
stopped at tree 48
for lr=0.05, depth = 2, the score is 0.5177892888341191
stopped at tree 35
for lr=0.05, depth = 3, the score is 0.5303810763777898
stopped at tree 29
for lr=0.05, depth = 4, the score is 0.5441669643340353
stopped at tree 26
for lr=0.05, depth = 5, the score is 0.5547240595055235
stopped at tree 57
for lr=0.1, depth = 1, the score is 0.516357576600008
stopped at tree 23
for lr=0.1, depth = 2, the score is 0.5168572907126091
stopped at tree 17
for lr=0.1, depth = 3, the score is 0.5283730980215416
stopped at tree 14
for lr=

In [16]:
    for lr in [0.01,0.05,0.1]:
        for depth in [1,2,3,4,5]:
            xgb = models.XGBOOST_QLIKE(train,val,cols,depth,lr)
            score = metrics.QLIKE(xgb,val["target"])
            

            print(f"for lr={lr}, depth = {depth}, the score is {score}")

stopped at tree 525
for lr=0.01, depth = 1, the score is 0.4972284886568703
stopped at tree 267
for lr=0.01, depth = 2, the score is 0.5052334532019582
stopped at tree 189
for lr=0.01, depth = 3, the score is 0.5396272007583186
stopped at tree 137
for lr=0.01, depth = 4, the score is 0.5906634625204243
stopped at tree 139
for lr=0.01, depth = 5, the score is 0.5999097557054984
stopped at tree 97
for lr=0.05, depth = 1, the score is 0.4991456709787314
stopped at tree 49
for lr=0.05, depth = 2, the score is 0.509683383766101
stopped at tree 36
for lr=0.05, depth = 3, the score is 0.5422824532861811
stopped at tree 26
for lr=0.05, depth = 4, the score is 0.5935842343360651
stopped at tree 26
for lr=0.05, depth = 5, the score is 0.6068755709684662
stopped at tree 47
for lr=0.1, depth = 1, the score is 0.4997938475008028
stopped at tree 24
for lr=0.1, depth = 2, the score is 0.511014118138871
stopped at tree 16
for lr=0.1, depth = 3, the score is 0.5513504382584083
stopped at tree 13
for lr

In [17]:
xg1 = models.XGBOOST_QLIKE(train,val,cols,1,0.01)
xg2 = models.XGBOOST(train,val,cols,1,0.01)
print(metrics.RMSE(xg1,val["target"]))
print(metrics.RMSE(xg2,val["target"]))

stopped at tree 525
stopped at tree 580
0.006868201011582088
0.006610429076311928


In [18]:
both = pd.concat([xg1,val],axis = 1,join='inner').dropna()
print(both.iloc[:,0].mean(), both.iloc[:,1].mean())

0.0112737 0.011109548164003918
